# 3.1.1 The coin flip\n
\n
This notebook visualizes the exact distribution of the normalized payoff after \(n\) fair coin flips.\n
\n
If each win is `+1` dollar and each loss is `-1` dollar, then the total payoff is\n
\n
$$S_n = \sum_{i=1}^n X_i, \qquad X_i \in \{-1, +1\}. $$\n
\n
To compare different games on the same scale, we normalize by the notional (the maximum money at stake), which is just `n` dollars here:\n
\n
$$Y_n = \frac{S_n}{n}. $$\n
\n
A subtlety: after normalization, the grid spacing is different for each `n` (`2/n`). That means raw point heights from the PMF are not directly comparable on the normalized axis.\n
\n
So the best comparison is a **density-style plot**, where each probability mass is divided by its bin width. Then the **area** under the spikes is the probability, and the larger-`n` distributions appear both narrower and taller, which is the correct picture.

In [ ]:
import os
from math import comb, sqrt
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")


def normalized_payoff_pmf(n: int):
    """Return support and exact PMF of S_n / n for n fair coin flips."""
    k = np.arange(n + 1)
    payoffs = (2 * k - n) / n
    probs = np.array([comb(n, int(i)) / (2**n) for i in k], dtype=float)
    return payoffs, probs


def pmf_to_density(x: np.ndarray, p: np.ndarray):
    """Convert discrete masses on an evenly spaced grid into heights with comparable area."""
    dx = x[1] - x[0]
    return dx, p / dx


short_x, short_p = normalized_payoff_pmf(20)
mid_x, mid_p = normalized_payoff_pmf(400)
long_x, long_p = normalized_payoff_pmf(2000)
short_dx, short_density = pmf_to_density(short_x, short_p)
mid_dx, mid_density = pmf_to_density(mid_x, mid_p)
long_dx, long_density = pmf_to_density(long_x, long_p)

print(f"20 games:   mean = 0.000, std = {1 / sqrt(20):.3f}")
print(f"400 games:  mean = 0.000, std = {1 / sqrt(400):.3f}")
print(f"2000 games: mean = 0.000, std = {1 / sqrt(2000):.3f}")
print(f"20 games:   normalized bin width = {short_dx:.3f}")
print(f"400 games:  normalized bin width = {mid_dx:.3f}")
print(f"2000 games: normalized bin width = {long_dx:.3f}")

In [ ]:
output_path = Path("../images/coin_flip_normalized_payoff.png")
output_path.parent.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(10.5, 4.2))

ax.bar(short_x, short_density, width=short_dx, color="#b44b3e", alpha=0.40, edgecolor="#b44b3e", linewidth=1.2, label="20 games", align="center")
ax.bar(mid_x, mid_density, width=mid_dx, color="#2a6fbb", alpha=0.40, edgecolor="#2a6fbb", linewidth=0.35, label="400 games", align="center")
ax.bar(long_x, long_density, width=long_dx, color="#2f8f5b", alpha=0.55, edgecolor="#2f8f5b", linewidth=0.2, label="2000 games", align="center")
ax.axvline(0, color="black", linewidth=1, alpha=0.5)
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(0, 19)
ax.set_xlabel("Normalized payoff  $S_n / n$")
ax.set_ylabel("Density")
ax.legend(frameon=True)
sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"Saved to {output_path}")